# 📓 Exemplo 08: Duas normais, antes e depois

**Missão 7** · [![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcoliveira-utfpr/change_climate/blob/main/code/notebooks_exemplo/08_duas_normais.ipynb)

> **Pergunta:** o clima de 1991-2020 é diferente do de 1961-1990? As folgas diminuíram?

### 🧭 Como usar este notebook
1. 💾 **Faça uma cópia:** menu **Arquivo → Salvar uma cópia no Drive**. Mova a cópia para a pasta
   `mudanca_climatica_br/notebooks/`. Assim você pode mudar tudo sem estragar o original.
2. ▶️ **Rode uma célula por vez** (botão ▶ ou **Shift + Enter**) e **leia os comentários** (as linhas com `#`).
3. 🧪 Os exemplos usam **Santa Helena-PR** e **Curitiba-PR**.
4. ✍️ No fim, a seção **Sua vez** pede para você **replicar o método para as capitais do Brasil**.

In [ ]:
# Instala as bibliotecas que não vêm no Colab (só precisa rodar uma vez por sessão)
!pip install agrometeorologiapy pymannkendall

In [ ]:
# Endereço da pasta de dados do projeto no GitHub (arquivos "raw" = texto puro).
# O pandas consegue ler um CSV direto de um endereço da internet.
URL_DADOS = 'https://raw.githubusercontent.com/fcoliveira-utfpr/change_climate/main/code/dados'

In [ ]:
import pandas as pd

# Lê as séries mensais (1961-2025) direto do GitHub.
#   parse_dates=['data'] -> a coluna 'data' é entendida como data
#   index_col='data'     -> a coluna 'data' vira o índice (o "nome" de cada linha)
serie_sh = pd.read_csv(f'{URL_DADOS}/santa_helena_mensal.csv', parse_dates=['data'], index_col='data')
serie_ct = pd.read_csv(f'{URL_DADOS}/curitiba_mensal.csv', parse_dates=['data'], index_col='data')

LATITUDE_SH = -24.86
LATITUDE_CT = -25.43

serie_sh.head()

## 1. Calcular as normais a partir da série

In [ ]:
# Recorta o período e tira a média de cada mês (todos os janeiros, todos os fevereiros...)
antes_sh = serie_sh.loc['1961':'1990']
normal_antes_sh = antes_sh.groupby(antes_sh.index.month).mean()

depois_sh = serie_sh.loc['1991':'2020']
normal_depois_sh = depois_sh.groupby(depois_sh.index.month).mean()

normal_antes_sh.round(1)

## 2. Classificar os dois períodos (Santa Helena)

In [ ]:
import agrometeorologiapy as amp

# As funções da biblioteca recebem listas: .tolist() transforma a coluna em lista
t_antes = normal_antes_sh['tmed'].tolist()
p_antes = normal_antes_sh['chuva'].tolist()
t_depois = normal_depois_sh['tmed'].tolist()
p_depois = normal_depois_sh['chuva'].tolist()

# Köppen
koppen_antes = amp.classificacao_koppen(t_antes, p_antes, LATITUDE_SH)
koppen_depois = amp.classificacao_koppen(t_depois, p_depois, LATITUDE_SH)

# Thornthwaite
etp_antes = amp.thornthwaite_mensal(pd.DataFrame({'T_media_C': t_antes}), lat=LATITUDE_SH)['ETP_mm_mes']
etp_depois = amp.thornthwaite_mensal(pd.DataFrame({'T_media_C': t_depois}), lat=LATITUDE_SH)['ETP_mm_mes']
th_antes = amp.classificacao_thornthwaite(p_antes, etp_antes, LATITUDE_SH)
th_depois = amp.classificacao_thornthwaite(p_depois, etp_depois, LATITUDE_SH)

print('1961-1990 -> Köppen:', koppen_antes['classe'], '| Thornthwaite:', th_antes['classe'])
print('1991-2020 -> Köppen:', koppen_depois['classe'], '| Thornthwaite:', th_depois['classe'])

## 3. A tabela das folgas: antes × depois

In [ ]:
tabela_sh = pd.DataFrame({
    '1961-1990': [min(t_antes) - 18, max(t_antes) - 22, min(p_antes) - 60, th_antes['Im'], th_antes['ETP_anual']],
    '1991-2020': [min(t_depois) - 18, max(t_depois) - 22, min(p_depois) - 60, th_depois['Im'], th_depois['ETP_anual']],
}, index=['Folga A/C (°C)', 'Folga a/b (°C)', 'Folga Af (mm)', 'Im (%)', 'ETP anual (mm)'])

# Diferença = depois menos antes
tabela_sh['Diferença'] = tabela_sh['1991-2020'] - tabela_sh['1961-1990']
tabela_sh.round(2)

> 🧠 Leia cada linha da coluna **Diferença**: a folga chegou mais perto de zero? Cruzou o zero? Se cruzou,
> a classe mudou, e isso deve aparecer na seção 2.

## 4. Curitiba
O mesmo código, trocando `serie_sh` por `serie_ct` e `LATITUDE_SH` por `LATITUDE_CT`. Repare como estamos
**repetindo** código; no Exemplo 10 vamos resolver isso com **funções**.

In [ ]:
antes_ct = serie_ct.loc['1961':'1990']
depois_ct = serie_ct.loc['1991':'2020']
normal_antes_ct = antes_ct.groupby(antes_ct.index.month).mean()
normal_depois_ct = depois_ct.groupby(depois_ct.index.month).mean()

t_antes = normal_antes_ct['tmed'].tolist()
p_antes = normal_antes_ct['chuva'].tolist()
t_depois = normal_depois_ct['tmed'].tolist()
p_depois = normal_depois_ct['chuva'].tolist()

print('1961-1990 -> Köppen:', amp.classificacao_koppen(t_antes, p_antes, LATITUDE_CT)['classe'])
print('1991-2020 -> Köppen:', amp.classificacao_koppen(t_depois, p_depois, LATITUDE_CT)['classe'])
print('Folga a/b: 1961-1990 =', round(max(t_antes) - 22, 2), '°C | 1991-2020 =', round(max(t_depois) - 22, 2), '°C')

## ✍️ Sua vez
1. Monte a tabela "antes × depois" para **10 capitais** (2 de cada região). Use `capitais/<uf>.csv` e a
   latitude da tabela `capitais_27.csv`.
2. Quantas mudaram de classe de Köppen? E de Thornthwaite?
3. Em quantas a folga do critério mais próximo **diminuiu**?

In [ ]:
# As capitais que VOCÊ extraiu na Missão 5 (exemplo 06) estão no seu Drive.
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/mudanca_climatica_br'

# Tabela das 27 capitais (uf, capital, regiao, lat, lon), montada por você
capitais = pd.read_csv(f'{BASE}/dados/capitais_27.csv')

# Exemplo: lendo a série de uma capital (sigla em minúsculas)
uf = 'df'
serie_capital = pd.read_csv(f'{BASE}/dados/capitais/{uf}.csv', parse_dates=['data'], index_col='data')
serie_capital.head()